# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [3]:
# TOOL 3 (Bonus): Word Counter

def count_words(text: str) -> dict:
    """Count words and characters in text."""
    try:
        words = text.split()
        return {"word_count": len(words), "char_count": len(text)}
    except Exception:
        return {"word_count": 0, "char_count": 0}

In [4]:
# TOOL 4 (Bonus): Quiz Q&A Lookup — answers questions from the quiz doc

import re

QUIZ_QA = [
    {"q": "Explain the concept of a stateful directed graph in agent pipelines. How does it differ from a simple linear pipeline?",
     "a": "A stateful directed graph is a workflow where each step can store and use information from previous steps, using nodes and edges to define data flow, supporting branching, looping, and decision-making. A linear pipeline follows a fixed sequence without remembering state or changing path."},
    {"q": "Describe the role of nodes and edges in an agent workflow. Give an example of each.",
     "a": "Nodes are individual tasks/actions (e.g. a Calculator Tool). Edges are connections between nodes that determine information flow (e.g. the path from query analysis to the calculator)."},
    {"q": "What is conditional routing in an agent system? Design a simple rule-based routing logic for three different query types.",
     "a": "Conditional routing directs a query to different tools based on intent. Example: 'calculate' -> Calculator Tool, 'keywords' -> Keyword Extraction Tool, everything else -> General Response."},
    {"q": "Why are cycles (loops) important in agent pipelines? Provide a use case where a retry loop is necessary.",
     "a": "Loops let an agent repeat a process until it succeeds, useful when tasks may fail. Example: retrying an API call after a temporary network failure instead of giving up immediately."},
    {"q": "Explain how a single-agent system can simulate multi-agent behavior internally.",
     "a": "A single agent can divide its work into roles — analyze the query, pick a tool, generate a response — each acting like a separate agent even though one system executes all of them."},
    {"q": "What are JSON schema tools? How do they help in structuring tool inputs and outputs?",
     "a": "JSON schema tools define a standard structure (required fields, types, formats) for data exchanged with tools, ensuring valid inputs and consistent, easy-to-interpret outputs."},
    {"q": "Compare sequential tool calls and parallel tool calls. When would you prefer one over the other?",
     "a": "Sequential calls run one after another when steps depend on each other. Parallel calls run independent tasks simultaneously, reducing response time when tasks are unrelated."},
    {"q": "How would you implement error handling in a tool-using agent? Provide at least two strategies.",
     "a": "1) try/except blocks that catch exceptions and return meaningful errors. 2) Automatic retry mechanisms that repeat failed operations. Logging errors also helps debugging."},
    {"q": "What is trajectory evaluation in agent systems? Why is it important beyond just checking final output?",
     "a": "Trajectory evaluation examines the full sequence of actions (decisions, tool calls, intermediate steps), not just the final answer, helping find mistakes and improve agent performance."},
    {"q": "Define task completion rate and cost metrics. How would you measure and optimize them in a real-world system?",
     "a": "Task completion rate = % of tasks successfully completed. Cost metrics = resources used (API calls, time, money). Measure across many tasks; optimize via better routing accuracy and fewer unnecessary tool calls."},
]

def match_quiz_question(query: str, threshold: float = 0.5):
    """Keyword-overlap match against the quiz Q&A bank. Returns the answer text or None."""
    q_words = set(re.findall(r"[a-zA-Z]{4,}", query.lower()))
    best_answer, best_score = None, 0.0
    for item in QUIZ_QA:
        item_words = set(re.findall(r"[a-zA-Z]{4,}", item["q"].lower()))
        overlap = len(q_words & item_words)
        score = overlap / max(1, min(len(q_words), len(item_words)))  # ratio vs the SHORTER set, so short queries can still match
        if score > best_score:
            best_score, best_answer = score, item["a"]
    return best_answer if best_score >= threshold else None

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [5]:
# AGENT FUNCTION (with bonus: improved routing + logging + extra tool)

import re

MATH_PATTERN = re.compile(r"[-+*/0-9]\s*[-+*/]\s*[-+*/0-9]")  # catches expressions even without the word 'calculate'

def log(msg: str):
    print(f"[agent-log] {msg}")

def agent(query: str):
    query_lower = query.lower()
    try:
        # --- improved routing: keyword synonyms + regex fallback for math ---
        if "calculate" in query_lower or "compute" in query_lower or MATH_PATTERN.search(query):
            log(f"routed to CALCULATOR for query: '{query}'")
            expr = re.sub(r"(calculate|compute)", "", query_lower).strip()
            res = calculator(expr)
            if res == "Error in calculation":
                log("calculator failed")
                return {"type": "error", "result": res}
            return {"type": "calculation", "result": res}

        elif "keyword" in query_lower:  # matches 'keyword' and 'keywords'
            log(f"routed to KEYWORD EXTRACTOR for query: '{query}'")
            return {"type": "keywords", "result": extract_keywords(query)}

        elif "count" in query_lower or "how many words" in query_lower:
            log(f"routed to WORD COUNTER for query: '{query}'")
            return {"type": "word_count", "result": count_words(query)}

        else:
            quiz_answer = match_quiz_question(query)
            if quiz_answer:
                log(f"routed to QUIZ Q&A LOOKUP for query: '{query}'")
                return {"type": "quiz_answer", "result": quiz_answer}
            log(f"routed to GENERAL RESPONSE for query: '{query}'")
            return {"type": "general", "result": f"You asked: '{query}'. This is a general response."}

    except Exception as e:
        log(f"ERROR: {e}")
        return {"type": "error", "result": str(e)}

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [9]:
# Pretty Output Helper

import json

TYPE_ICONS = {
    "calculation": "",
    "keywords": "",
    "word_count": "",
    "quiz_answer": "",
    "general": "",
    "error": "",
}

def display_response(query: str, response: dict):
    icon = TYPE_ICONS.get(response["type"], "")
    print(f"\n  Query : {query}")
    print(f"{icon}  Type  : {response['type']}")
    print(f"  Result:")
    print(json.dumps(response["result"], indent=2) if isinstance(response["result"], (dict, list)) else f"    {response['result']}")
    print("─" * 60)

In [10]:
#  Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    display_response(q, agent(q))

[agent-log] routed to CALCULATOR for query: 'Calculate 20 + 5'

  Query : Calculate 20 + 5
  Type  : calculation
  Result:
    25
────────────────────────────────────────────────────────────
[agent-log] routed to KEYWORD EXTRACTOR for query: 'Extract keywords from Artificial Intelligence is transforming industries'

  Query : Extract keywords from Artificial Intelligence is transforming industries
  Type  : keywords
  Result:
[
  "extract",
  "industries",
  "intelligence",
  "artificial",
  "keywords"
]
────────────────────────────────────────────────────────────
[agent-log] routed to GENERAL RESPONSE for query: 'What is machine learning?'

  Query : What is machine learning?
  Type  : general
  Result:
    You asked: 'What is machine learning?'. This is a general response.
────────────────────────────────────────────────────────────


In [11]:
#  Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        print("👋 Session ended.")
        break
    display_response(user_input, agent(user_input))

Enter query (type 'exit' to stop): compute 15 * 3
[agent-log] routed to CALCULATOR for query: 'compute 15 * 3'

  Query : compute 15 * 3
  Type  : calculation
  Result:
    45
────────────────────────────────────────────────────────────
Enter query (type 'exit' to stop): extract keywords from Artificial Intelligence is transforming industries
[agent-log] routed to KEYWORD EXTRACTOR for query: 'extract keywords from Artificial Intelligence is transforming industries'

  Query : extract keywords from Artificial Intelligence is transforming industries
  Type  : keywords
  Result:
[
  "extract",
  "industries",
  "intelligence",
  "artificial",
  "keywords"
]
────────────────────────────────────────────────────────────
Enter query (type 'exit' to stop): count words in this sentence
[agent-log] routed to WORD COUNTER for query: 'count words in this sentence'

  Query : count words in this sentence
  Type  : word_count
  Result:
{
  "word_count": 5,
  "char_count": 28
}
─────────────────────